## Transform Dataset: Alignment and Stationarity

Aligns all raw series to a common trading-day index, flags forward-filled values, and converts each series to a stationary form (first difference or log return).

In [1]:
import sys
import pickle
import pandas as pd
import numpy as np
from openpyxl.styles import PatternFill, Font, Alignment

sys.path.append('..')
from modules.source import Sources, START, END, RAW_DATA_PATH, OUTPUT_FOLDER_PATH, parquet_monthly, parquet_daily, parquet_raw_levels
from modules.helpers import combine_trading_days

with open(f"{RAW_DATA_PATH}/data_frames.pkl", "rb") as f:
   data_frames = pickle.load(f)["data_frames"]

### Publication-lag adjustment

Palm Oil and CPI are adjusted back to their publication date (~2 months), then trimmed to the analysis window.

In [ ]:
for n in (Sources.palm_oil_global, Sources.cpi_inflation_yoy):
   data_frames[n] = data_frames[n].shift(2, freq="BME").loc[START:END]

### Align trading days and forward-fill

Now we forward fill all data and adjust for timezone difference.

In [4]:
monthly_series: list[str] = [Sources.palm_oil_global, Sources.FFR_midpoint, Sources.OPR, Sources.cpi_inflation_yoy]
daily_freq_indices: list[str]  = [n for n in data_frames if n not in monthly_series]
all_trading_days:   pd.Index   = combine_trading_days([data_frames[n] for n in daily_freq_indices])
original_dates: dict[pd.Index] = {n: df.index for n, df in data_frames.items()}

shift_timezone = [Sources.UST_10Y, Sources.VIX, Sources.USDMYR, Sources.brent_oil, Sources.DXY, Sources.FFR_midpoint]

for n in monthly_series: # Data could be published on non-trading days. Fill all days then trim to trading days.
   data_frames[n] = (
      data_frames[n]
      .reindex(pd.date_range(START, END, freq="D"))
      .ffill()
      .reindex(index=all_trading_days)
   )

for n in daily_freq_indices:
   data_frames[n] = data_frames[n].reindex(index=all_trading_days).ffill()

is_ffilled = {n: pd.Series(~all_trading_days.isin(original_dates[n]), index=all_trading_days) for n in data_frames}

for n in shift_timezone: # timezone adjustment, not needed for Palm Oil since its monthly published data
   data_frames[n] = data_frames[n].shift(1)
   is_ffilled[n] = is_ffilled[n].shift(1)

all_trading_days = all_trading_days[1:] # drop the 1st index where there is no data

### Combine into MultiIndex frames

Create MultiIndex DataFrames to store forward-filled flags in the `is_ffilled` column, and combine all DataFrames.

In [16]:
df_prices = pd.concat({n: data_frames[n].iloc[:, 0] for n in data_frames}, axis=1).reindex(all_trading_days)
df_flags = pd.DataFrame(is_ffilled).reindex(all_trading_days).where(df_prices.notna())

df_combined_daily = pd.concat({'price': df_prices, 'is_ffilled': df_flags}, axis=1).swaplevel(0, 1, axis=1)
df_combined_monthly = df_combined_daily.resample("ME").last()

### Export master workbook

Apply formatting to all columns and forward-filled values for daily and monthly data and export to Excel (output folder).

In [ ]:
def export_master_xlsx(sheets_data: dict[str, tuple[pd.DataFrame]], path: str):
   with pd.ExcelWriter(path, engine="openpyxl") as writer:
      for sheet_name, (prices_df, flags_df) in sheets_data.items():
         prices_export = prices_df.set_index(prices_df.index.date)
         prices_export.to_excel(writer, sheet_name=sheet_name)
         ws = writer.sheets[sheet_name]

         for r, c in zip(*np.where(flags_df.values == True)):
            cell = ws.cell(row=r + 2, column=c + 2)
            cell.fill, cell.font = PatternFill("solid", "F3F4F6"), Font(color="6B7280") # Gray fill & font

         for cell in ws[1]: 
            n = len(str(cell.value or ""))
            ws.column_dimensions[cell.column_letter].width = n if n > 8 else n + 3
            cell.alignment = Alignment(horizontal="right")
         ws.column_dimensions["A"].width = 11

def get_price_and_flags(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
   return df.xs("price", axis=1, level=1), df.xs("is_ffilled", axis=1, level=1)

export_master_xlsx(
   sheets_data={
      "daily": get_price_and_flags(df_combined_daily),
      "monthly": get_price_and_flags(df_combined_monthly)
   },
   path=f"{OUTPUT_FOLDER_PATH}/price_data_master.xlsx"
)

### Transform to stationary and save

First-difference the level/rate series (`first_diff_var`) and log-difference the price series (`log_ret_var`); save daily and monthly parquet files. Also saved raw prices for charting.

In [ ]:
first_diff_var = [Sources.VIX, Sources.FFR_midpoint, Sources.UST_10Y, Sources.MGS_10Y, Sources.cpi_inflation_yoy, Sources.OPR]
log_ret_var = [Sources.palm_oil_global, Sources.USDMYR, Sources.DXY, Sources.brent_oil, Sources.KLCI, Sources.financials, Sources.plantation,Sources.REITs, Sources.technology, Sources.energy, Sources.industrial_products]

def transform_to_stationary(df_raw: pd.DataFrame) -> pd.DataFrame:
   df_trans = df_raw.copy()
   df_trans.loc[:, (first_diff_var, "price")] = df_trans.loc[:, (first_diff_var, "price")].astype(float).diff()
   df_trans.loc[:, (log_ret_var, "price")] = np.log(df_trans.loc[:, (log_ret_var, "price")].astype(float)).diff()
   return df_trans.iloc[1:].rename(columns={'price': 'change'}, level=1)

df_combined_daily.to_parquet(parquet_raw_levels)

df_daily = transform_to_stationary(df_combined_daily)
df_daily.to_parquet(parquet_daily)

df_monthly = transform_to_stationary(df_combined_monthly)
df_monthly.to_parquet(parquet_monthly)